# 2.1 Train Validation Test 切分

這份 Notebook 示範如何正確切分 train、validation、test，並用簡單 DNN 驗證流程。

## 1. 載入套件

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.keras.utils.set_random_seed(42)

## 2. 載入資料與觀察類別比例

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data.values.astype('float32')
y = data.target.values.astype('int64')

pd.Series(y).value_counts(normalize=True).rename(index=dict(enumerate(data.target_names))).to_frame('ratio')

## 3. 切分 train / validation / test

先切出 test，再從 train 中切出 validation。分類任務使用 `stratify` 維持類別比例。

In [ ]:
x_train_full, x_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
x_train, x_valid, y_train, y_valid = train_test_split(
    x_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

summary = pd.DataFrame({
    'dataset': ['train', 'valid', 'test'],
    'n_samples': [len(y_train), len(y_valid), len(y_test)],
    'positive_ratio': [y_train.mean(), y_valid.mean(), y_test.mean()]
})
summary

## 4. 標準化資料

Scaler 只 fit 訓練集，再 transform validation/test。

In [ ]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train).astype('float32')
x_valid = scaler.transform(x_valid).astype('float32')
x_test = scaler.transform(x_test).astype('float32')

print(x_train.shape, x_valid.shape, x_test.shape)

## 5. 建立並訓練模型

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(x_train.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
history = model.fit(x_train, y_train, validation_data=(x_valid, y_valid), epochs=40, batch_size=32, verbose=0)

## 6. 評估 train / validation / test

In [ ]:
results = [
    model.evaluate(x_train, y_train, verbose=0, return_dict=True),
    model.evaluate(x_valid, y_valid, verbose=0, return_dict=True),
    model.evaluate(x_test, y_test, verbose=0, return_dict=True),
]
pd.DataFrame(results, index=['train', 'valid', 'test'])

## 7. 小結

資料切分的目的，是讓模型訓練、調參與最終評估各自有明確角色。